**Table of contents**<a id='toc0_'></a>    
- [<u>**Objective:**</u> Create a model pipeline, and finalize train_model.py](#toc1_1_1_1_1_)    
- [Import Necessary Modules](#toc2_)    
- [Import Dataset and Perform train-tests Split](#toc3_)    
- [Grouping Features](#toc4_)    
- [Ordinal Mappings](#toc5_)    
- [Few Things to Verify](#toc6_)    
- [Creating Pipeline and Prediction Model](#toc7_)    
  - [Encoding Transformers](#toc7_1_)    
    - [Ordinal Encoders](#toc7_1_1_)    
    - [Nominal Encoders](#toc7_1_2_)    
  - [Create Pipelines](#toc7_2_)    
    - [Create branch_preprocessor and base_imputation_and_engineering_pipeline pipelines](#toc7_2_1_)    
    - [Create Preprocessing Pipeline](#toc7_2_2_)    
    - [Inspect the Preprocessor Output](#toc7_2_3_)    
    - [Create Model Pipeline](#toc7_2_4_)    
    - [Target Transformation](#toc7_2_5_)    
  - [Train and Predict Using the Model](#toc7_3_)    
    - [Make the Complete Model](#toc7_3_1_)    
    - [Make Predictions and Save it](#toc7_3_2_)    
    - [Kaggle Submission and Analysis](#toc7_3_3_)    
    - [Serialize the Model](#toc7_3_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

##### <a id='toc1_1_1_1_1_'></a>[<u>**Objective:**</u> Create a model pipeline, and finalize train_model.py](#toc0_)

# <a id='toc2_'></a>[Import Necessary Modules](#toc0_)

In [2]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
#from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
from xgboost import XGBRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.metrics import root_mean_squared_error
import joblib


sys.path.append(os.path.abspath(os.path.join("..")))

from src.features.preprocessing import (
    remove_training_outliers,
    NeighborhoodLotFrontageImputer,
    MasVnrImputer,
    StructuralImputer,
    AmesFeatureEngineer
)

# <a id='toc3_'></a>[Import Dataset and Perform train-tests Split](#toc0_)

In [12]:
# Load the raw train.csv and test.csv datasets
train_df = pd.read_csv('../data/raw/train.csv')
test_df = pd.read_csv('../data/raw/test.csv')

# Separate target and predictors and remove the SalePrice column from the feature matrix. 
y =train_df['SalePrice']
X = train_df.drop(columns="SalePrice") # Ensures readability


# Perform a train/validation split on the training data.
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True, stratify=None)

X_train, y_train = remove_training_outliers(X_train, y_train)

# X_train = AmesFeatureEngineer().transform(X_train)
# Forgetting to comment this line would result in c:\Users\Shiv Shankar Dubey\Desktop\HousePrices\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['CentralAir']. At least one non-missing value is needed for imputation with strategy='most_frequent'.warnings.warn(. during model training. Because the dictionary excepts "Y" and 'N' but gets 0 and 1 so puts everything to null. And this situation cannot be handled by any imputers as they require at least one not missing value. 
# X_train.shape

**Explanation of Code Above**:

* os.path.join('..'):The double dots .. represent the "parent directory" relative to where your code is running. os.path.join formats this correctly depending on your operating system (using / for Linux/Mac or \ for Windows).

* os.path.abspath(...):This converts the relative path (..) into an absolute path (e.g., turning .. into a full path like /Users/username/projects/my_ml_pipeline). This ensures the path remains correct even if you change working directories later.

* sys.path.append(...):sys.path is a standard list of directory paths where Python looks whenever you type import my_module. By running .append(), you add your parent directory to this list.

* When we run a Jupyter Notebook, Python thinks the notebooks/ folder is the center of the universe. It looks around for a folder named src/ inside notebooks/, so it doesn't find it, and crashes. In such cases Python will throw a ModuleNotFoundError.

# <a id='toc4_'></a>[Grouping Features](#toc0_)

In [13]:
continuous_features = [
    'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtUnfSF', 
    'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 
    'ScreenPorch', '3SsnPorch', 'PoolArea', 'MiscVal', 
    'GrLivArea', '1stFlrSF', '2ndFlrSF', 'TotalBsmtSF', 
    'LowQualFinSF', 'BsmtFinSF1', 'BsmtFinSF2', 'TotalUsableSF', 
    'BsmtFinishedRatio', 'OverallQual_x_TotalUsableSF'
    ]

discrete_features = [
    'TotalBathrooms', 'KitchenAbvGr', 'TotRmsAbvGrd', 
    'Fireplaces', 'BedroomAbvGr', 'MoSold', 'GarageCars'
    ]

elapsed_time_features = ['HouseAge', 'YearsSinceRemodel', 'GarageAge']

ordinal_features = [
    'OverallQual', 'OverallCond', 'ExterQual', 'ExterCond', 
    'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 
    'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC', 
    'Fence', 'GarageFinish', 'LotShape', 'LandSlope', 
    'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
    'Utilities', 'Functional', 'PavedDrive'
    ]

nominal_high_card_features = [
    'Neighborhood', 'Exterior1st', 'Exterior2nd', 'MSSubClass' 
]

nominal_low_card_features = [
    'MSZoning', 'Street', 'Alley', 'BldgType', 'HouseStyle', 
    'RoofStyle', 'Foundation', 'GarageType', 'SaleCondition', 
    'LotConfig', 'LandContour', 'MasVnrType', 'Heating', 
    'Electrical', 'MiscFeature', 'Condition1', 'Condition2', 
    'RoofMatl', 'SaleType'
]

binary_features = ['CentralAir', 'HasGarage', 'HasBsmt']


numeric_features = (
    continuous_features
    + discrete_features
    + elapsed_time_features
)


all_features = (
    numeric_features
    + ordinal_features
    + nominal_high_card_features
    + nominal_low_card_features
    + binary_features
)

excluded_features = {
    "Id",
    "FullBath",
    "HalfBath",
    "BsmtFullBath",
    "BsmtHalfBath",
    "YearRemodAdd",
    "YrSold",
    'YearBuilt', 
    'GarageYrBlt'
}

In [14]:
# Later on added due to problem in making ordinal_encooder for OverallQual and OverallCond. So, we will treat them as numeric features.
numeric_features += [
    "OverallQual",
    "OverallCond",
]

ordinal_features.remove("OverallQual")
ordinal_features.remove("OverallCond")

# <a id='toc5_'></a>[Ordinal Mappings](#toc0_)

In [15]:
quality_scale = [
    "None",
    "Po",
    "Fa",
    "TA",
    "Gd",
    "Ex",
]

garage_finish_scale = [
    "None",
    "Unf",
    "RFn",
    "Fin",
]

basement_finish_scale = [
    "None", 
    "Unf", 
    "LwQ", 
    "Rec", 
    "BLQ", 
    "ALQ", 
    "GLQ"
    ]

exposure_scale = [
    "None",
    "No",
    "Mn",
    "Av",
    "Gd",
]



ordinal_mappings = {
    'ExterQual': quality_scale,
    'ExterCond': quality_scale,
    'BsmtQual': quality_scale,
    'BsmtCond': quality_scale,
    'HeatingQC': quality_scale,
    'KitchenQual': quality_scale,
    'FireplaceQu': quality_scale,
    'GarageQual': quality_scale,
    'GarageCond': quality_scale,
    'PoolQC': quality_scale,
    'Fence': ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"],
    'GarageFinish': garage_finish_scale,
    'BsmtExposure': exposure_scale,
    'LotShape': ["IR3", "IR2", "IR1", "Reg"],
    'Utilities': ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    'LandSlope': ["Sev", "Mod", "Gtl"],
    'BsmtFinType1': basement_finish_scale,
    'BsmtFinType2': basement_finish_scale,
    'Functional': ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    'PavedDrive': ["N", "P", "Y"]
}

# binary_mappings = {
#     'CentralAir': {'N': 0, 'Y': 1}
# }
# No use of binary mapping as this is already handled in the AmesFeatureEngineer

# <a id='toc6_'></a>[Few Things to Verify](#toc0_)

In [7]:
print(train_df[nominal_high_card_features].nunique().sort_values())

Exterior1st     15
MSSubClass      15
Exterior2nd     16
Neighborhood    25
dtype: int64


In [8]:
print(train_df[nominal_low_card_features].nunique().sort_values())

Street           2
Alley            2
MasVnrType       3
LandContour      4
MiscFeature      4
LotConfig        5
MSZoning         5
BldgType         5
Electrical       5
GarageType       6
Foundation       6
RoofStyle        6
Heating          6
SaleCondition    6
HouseStyle       8
Condition2       8
RoofMatl         8
Condition1       9
SaleType         9
dtype: int64


In [9]:
assert len(all_features) == len(set(all_features))

In [10]:
missing_features = set(X_train.columns) - {"Id"} - set(all_features)
print(missing_features)

# Original columns intentionally excluded from the model.
# Their information is captured by engineered features.
#
# FullBath + HalfBath + BsmtFullBath + BsmtHalfBath
#     - TotalBathrooms
#
# YearRemodAdd
#     - YearsSinceRemodel
#
# YrSold
#     - HouseAge, GarageAge, YearsSinceRemodel

{'HalfBath', 'YearBuilt', 'BsmtFullBath', 'BsmtHalfBath', 'YearRemodAdd', 'YrSold', 'GarageYrBlt', 'FullBath'}


In [11]:
assert (
    set(X_train.columns) - excluded_features
    == set(all_features)
)

AssertionError: 

# <a id='toc7_'></a>[Creating Pipeline and Prediction Model](#toc0_)

## <a id='toc7_1_'></a>[Encoding Transformers](#toc0_)

### <a id='toc7_1_1_'></a>[Ordinal Encoders](#toc0_)

**Important Things to keep in mind:**
* Ordinal Encoder
    * Categories must be supplied in the same order as ordinal_features. The order of feature lists inside ordinal_categories must match the exact column order of the dataset you pass to .fit() or .transform(). 


In [16]:
# from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder

ordinal_categories = [
    ordinal_mappings[feature] for feature in ordinal_features
]

ordinal_encoder = OrdinalEncoder( 
    categories=ordinal_categories, # By default it sorts alphabetically, so passing custom list is better
    handle_unknown='use_encoded_value', # See the unknown_value parameter.
    unknown_value=-1, #this is the value that will be used for any unknown category encountered during transform. 
    encoded_missing_value=-1, # used to encode missing values in the input data. 
)

### <a id='toc7_1_2_'></a>[Nominal Encoders](#toc0_)

In [17]:
# Low Cardinality Encoder (One-Hot)
nominal_low_card_encoder = OneHotEncoder(
    handle_unknown='ignore', # Ignore by placing 0 across all one hot columns for that feature.
    sparse_output=False # Return a numpy array instead of skicki compressed sparse matrix.
)

# High Cardinality Encoder (Target Encoding)
nominal_high_card_encoder = TargetEncoder(
    target_type='continuous', # Tells target is continuous
    smooth='auto', # A mathematical function invoked to include a factor of overall average, whose weight incereases as the representation of a category in the dataset decreases.
    cv=5 # Cross validation, splits data into 5 parts and gets the average of this part using other 4 parts.
)  # By default uses global mean for unseen categories

## <a id='toc7_2_'></a>[Create Pipelines](#toc0_)

* A Pipeline executes steps sequentially (one after another), whereas a ColumnTransformer executes steps in parallel across different columns (side-by-side)

### <a id='toc7_2_1_'></a>[Create branch_preprocessor and base_imputation_and_engineering_pipeline pipelines](#toc0_)

In [18]:
# from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import StandardScaler
# from sklearn.compose import ColumnTransformer

base_imputation_and_engineering_pipeline = Pipeline(steps=[
    ('structural_imputer', StructuralImputer()),
    ('lot_frontage_imputer', NeighborhoodLotFrontageImputer()),
    ('masvnr_imputer', MasVnrImputer()),
    ('feature_engineer', AmesFeatureEngineer(excluded_features, {'N': 0, 'Y': 1}))
    #('printer', print(X_train['CentralAir'].describe()))
])

numeric_pipeline = Pipeline(steps=[
    ('safety_imputer', SimpleImputer(strategy='median'))
])

binary_pipeline = Pipeline(steps=[
    ('safety_imputer', SimpleImputer(strategy='most_frequent'))
])

branch_preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal', ordinal_encoder, ordinal_features),
        ('nominal_low', nominal_low_card_encoder, nominal_low_card_features),
        ('nominal_high', nominal_high_card_encoder, nominal_high_card_features),
        ('numeric', numeric_pipeline, numeric_features),
        ('binary', binary_pipeline, binary_features)
    ],
    remainder='drop' # Drops other columns that are not provided in ordinal_features, nominal_low_card_features etc.
)

### <a id='toc7_2_2_'></a>[Create Preprocessing Pipeline](#toc0_)

In [19]:
preprocessor = Pipeline(steps=[
    # Global transformations on the entire dataset
    ('base_processing', base_imputation_and_engineering_pipeline), # First handling structural missingness, anamolies and then creating features.
    
    ('branching', branch_preprocessor)    # Performing encoding, and imputations                         
])

### <a id='toc7_2_3_'></a>[Inspect the Preprocessor Output](#toc0_)

In [20]:
# Fit and transform the training data
# y_train is required here because TargetEncoder needs it to calculate means
X_train_processed = preprocessor.fit_transform(X_train, y_train)

# Verify the structural changes
print("Original X_train shape:", X_train.shape)
print("Processed X_train shape:", X_train_processed.shape)

# Check for any unexpected /accidental NaN values that might have slipped through
# import numpy as np
print("Total missing values after preprocessing:",  pd.isnull(X_train_processed).sum())

Original X_train shape: (1156, 80)
Processed X_train shape: (1156, 168)
Total missing values after preprocessing: 0


In [ ]:
# X_train.columns
# preprocessor.named_steps['branching'].get_feature_names_out()

### <a id='toc7_2_4_'></a>[Create Model Pipeline](#toc0_)

In [ ]:
# from sklearn.pipeline import Pipeline
# from sklearn.linear_model import Ridge

# # We put the preprocessor and the ML algorithm together into one final object
# model_pipeline = Pipeline(steps=[
#     ("preprocessor", preprocessor), 
#     ("model", Ridge(alpha=1.0, random_state=42)) 
# ])

In [21]:
# from sklearn.pipeline import Pipeline
# from xgboost import XGBRegressor

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=100, #  Tells XGBoost to build exactly 100 sequential decision trees
        random_state=42, 
        n_jobs=-1 #  Tells Python to use all available CPU cores on computer to train the model in parallel and faster way.
    ))
])

### <a id='toc7_2_5_'></a>[Target Transformation](#toc0_)

In [22]:
# from sklearn.compose import TransformedTargetRegressor

# Wrap the entire modeling pipeline (preprocessor + model) into the TransformedTargetRegressor
baseline_model = TransformedTargetRegressor(
    regressor=model_pipeline,
    func=np.log1p,       # Applies ln(1+y) to the target before training
    inverse_func=np.expm1 # Applies exp(y)-1 to the predictions before outputting
)


## <a id='toc7_3_'></a>[Train and Predict Using the Model](#toc0_)

### <a id='toc7_3_1_'></a>[Make the Complete Model](#toc0_)

In [ ]:
# from sklearn.metrics import root_mean_squared_error

# Fit the entire pipeline (Preprocessing + Target Transformation + XGBoost)
baseline_model.fit(X_train, y_train)

# Generate predictions on the training set to check baseline performance
train_predictions = baseline_model.predict(X_train)


    
# Calculate the Baseline Metric (RMSE)
print("Baseline Training RMSE: $", root_mean_squared_error(y_train, train_predictions))
print("Validation RMSLE:", root_mean_squared_log_error(y_train, train_predictions))

print()

print("Evaluating model on validation set...")
valid_preds = baseline_model.predict(X_valid)
rmsle = root_mean_squared_log_error(y_valid, valid_preds)
print(f"Validation RMSLE: {rmsle:.5f}")
print()

Baseline Training RMSE: $ 4496.6796875
Validation RMSLE: 0.023424098268151283
Evaluating model on validation set...
Validation RMSLE: 0.14923


**Oops! I think my model just overfitted!**

### <a id='toc7_3_2_'></a>[Make Predictions and Save it](#toc0_)

In [24]:
# Generate predictions on the unseen test data
test_predictions = baseline_model.predict(test_df)

# Format the submission dataframe
# Kaggle expects exactly two columns: 'Id' and 'SalePrice'
submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': test_predictions
})

# Export to CSV
submission.to_csv('baseline_submission.csv', index=False)

### <a id='toc7_3_3_'></a>[Kaggle Submission and Analysis](#toc0_)

Kaggle is calculating this:$$\text{Score} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (\ln(y_i) - \ln(\hat{y}_i))^2}$$

My Score: 0.14388

A training error of $3,866 translates to a log-error of roughly 0.02 on the training set.

Your Kaggle score on the unseen test set is 0.14.

That massive gap between 0.02 and 0.14 confirms that the complex tree-ensemble memorized the training data and struggled to generalize.

### <a id='toc7_3_4_'></a>[Serialize the Model](#toc0_)

In [ ]:
# import joblib
# import os

# Define the save path 
model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'xgboost_baseline_v2.joblib') 

# Version 1, used StandardScaler while Version 2 does not, but the Kaggle score is almost same 

# Serialize and save the model
joblib.dump(baseline_model, model_path)
print("Model successfully serialized and saved to:", model_path)

Model successfully serialized and saved to: ../models\xgboost_baseline_v2.joblib
